In [3]:
import requests
import torch
import re
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [4]:
ds = load_dataset("cardiffnlp/tweet_eval", "irony")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    784 non-null    object
 1   label   784 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 12.4+ KB


In [5]:
test['label'] = test['label'].apply(lambda x: 'ironic' if x == 1 else 'sincere')

labels = test['label'].unique()

test

,text,label
0,@user Can U Help?||More conservatives needed o...,sincere
1,"Just walked in to #Starbucks and asked for a ""...",ironic
2,#NOT GONNA WIN,sincere
3,@user He is exactly that sort of person. Weirdo!,sincere
4,So much #sarcasm at work mate 10/10 #boring 10...,ironic
...,...,...
779,"If you drag yesterday into today, your tomorro...",sincere
780,Congrats to my fav @user & her team & my birth...,sincere
781,@user Jessica sheds tears at her fan signing e...,sincere
782,#Irony: al jazeera is pro Anti - #GamerGate be...,ironic


In [6]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_18128\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


50860032

In [7]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [8]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
        {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
        {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
    ]
    
    start_time = time.time()

    try:
        response = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": True,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 3100
            }
        })
        response_time = time.time() - start_time
        vram_usage = get_gpu_memory_usage()
        ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
        response = response.json()
        response_text = response['message'].get('thinking', '') if 'message' in response else ''
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return "error", {}, 0, 0, 0, 0, f"API Error: {e}"
    
    if 'message' in response and 'content' in response['message']:
        classification_text = response['message']['content'].lower()
        print("Response fields:", ', '.join(response.keys()))
        print(response)
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": response_text + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": False,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower() if 'message' in response2 and 'content' in response2['message'] else ''
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}
    
    if all(count == label_counts[labels[0]] for count in label_counts.values()):
        content = 'error'
    else:
        content = max(label_counts, key=label_counts.get)
    
    print(f"Text: {text}")
    print(f"Response: {content}")
    
    return content, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [9]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_18128\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Response fields: model, created_at, message, done_reason, done, total_duration, load_duration, prompt_eval_count, prompt_eval_duration, eval_count, eval_duration
{'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-08T03:28:43.1463548Z', 'message': {'role': 'assistant', 'content': "The text expresses genuine frustration with needing more conservatives for #TSU and offering payment for posting. There's no irony or double meaning here; it's clear and direct.\n\n**Answer:** Sincere", 'thinking': 'Okay, so I need to figure out how to classify the given text into one of the provided labels for sentiment analysis. The task is to determine whether the text is "Sincere," "Ironic," or something else. Let me break down each part step by step.\n\nFirst, looking at the text: "@user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!" The user starts with a greeting and then says they need more conservatives for #TSU, which I assume is a hashtag related to student u

In [10]:
test.to_csv('results/deepseekR1_ZS_binary3.csv', index=False)
test

,text,label,prediction,label_counts,response_time,vram_usage,ram_usage,total_time,response_text
0,@user Can U Help?||More conservatives needed o...,sincere,sincere,"{'sincere': 1, 'ironic': 0}",9.114749,2261,89.460938,7.061687,"Okay, so I need to figure out how to classify ..."
1,"Just walked in to #Starbucks and asked for a ""...",ironic,ironic,"{'sincere': 1, 'ironic': 2}",5.911756,2220,89.457031,3.869954,"Alright, let's break this down step by step. F..."
2,#NOT GONNA WIN,sincere,ironic,"{'sincere': 0, 'ironic': 1}",5.766371,2226,89.882812,3.739546,"Alright, let's tackle this query step by step...."
3,@user He is exactly that sort of person. Weirdo!,sincere,ironic,"{'sincere': 0, 'ironic': 1}",7.076030,2230,90.332031,5.056216,"Okay, so I need to figure out how to classify ..."
4,So much #sarcasm at work mate 10/10 #boring 10...,ironic,error,"{'sincere': 0, 'ironic': 0}",7.820712,2203,91.074219,5.777532,"Alright, let's break this down step by step. T..."
...,...,...,...,...,...,...,...,...,...
779,"If you drag yesterday into today, your tomorro...",sincere,ironic,"{'sincere': 1, 'ironic': 2}",4.047705,2434,78.535156,2.009417,"Alright, so I need to classify this tweet base..."
780,Congrats to my fav @user & her team & my birth...,sincere,sincere,"{'sincere': 1, 'ironic': 0}",3.499084,2405,78.183594,1.466268,"Alright, so I need to classify this tweet into..."
781,@user Jessica sheds tears at her fan signing e...,sincere,sincere,"{'sincere': 1, 'ironic': 0}",3.513049,2429,78.687500,1.461032,"Alright, so I need to classify this tweet into..."
782,#Irony: al jazeera is pro Anti - #GamerGate be...,ironic,sincere,"{'sincere': 2, 'ironic': 1}",5.064395,2442,78.480469,3.014007,"Okay, so I need to figure out how to classify ..."


In [11]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.561224
F1 score: 0.598112
Precision: 0.647613
Recall: 0.561224


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 5.444124890833485
Average VRAM usage: 2185.8380102040815
Average RAM usage: 80.30652104591837
Average total time: 3.4056455395408167


In [13]:
# save results to txt
with open('results/deepseekR1_ZS_binary3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')
    